# Домашнє завдання: Внесення оновлень в БД і робота з транзакціями

Це ДЗ передбачене під виконання на локальній машині. Виконання з Google Colab буде суттєво ускладнене.

## Підготовка
1. Переконайтесь, що у вас встановлены необхідні бібліотеки:
   ```bash
   pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv
   ```

2. Створіть файл `.env` з параметрами підключення до бази даних classicmodels. Базу даних ви можете отримати через

  - docker-контейнер згідно існтрукції в [документі](https://www.notion.so/hannapylieva/Docker-1eb94835849480c9b2e7f5dc22ee4df9), також відео інструкції присутні на платформі - уроки "MySQL бази, клієнт для роботи з БД, Docker і ChatGPT для запитів" та "Як встановити Docker для роботи з базами даних без терміналу"
  - або встановивши локально цю БД - для цього перегляньте урок "Опціонально. Встановлення MySQL та  БД Сlassicmodels локально".
  
  Приклад `.env` файлу ми створювали в лекції. Ось його обовʼязкове наповнення:
    ```
    DB_HOST=your_host
    DB_PORT=3306 або 3307 - той, який Ви налаштували
    DB_USER=your_username
    DB_PASSWORD=your_password
    DB_NAME=classicmodels
    ```
  Якщо ви створили цей файл під час перегляду лекції - **новий створювати не треба**. Замініть лише назву БД, або пропишіть назву в коді створення підключення (замість отримання назви цільової БД зі змінних оточення). Але переконайтесь, що до `.env` файл лежить в тій самій папці, що і цей ноутбук.

  **УВАГА!** НЕ копіюйте скрит для **створення** `.env` файлу. В лекції він наводиться для прикладу. І давалось пояснення, що в реальних проєктах ми НІКОЛИ не пишемо доступи до бази в коді. Копіювання скрипта для створення `.env` файлу сюди в ДЗ буде вважатись грубою помилкою і ми зніматимемо бали.

3. Налаштуйте підключення через SQLAlchemy до БД за прикладом в лекції.

Рекомендую вивести (відобразити) змінну engine після створення. Вона має бути не None! Якщо None - значить у Вас не підтягнулись налаштування з .env файла.

Ви також можете налаштувати параметри підключення до БД без .env файла, просто прописавши текстом в відповідних місцях. Це - не рекомендований підхід.


## Завдання

### Завдання 1: Оновлення інформації про клієнта (2 бали)

**Створіть функцію для оновлення контактної інформації клієнта за його номером** з наступними можливостями:
- Оновлення телефону клієнта
- Оновлення email (якщо поле існує в таблиці)

Опціонально, якщо вам хочеться більше практики:
- Логування змін в окрему таблицю

Використайте підхід з параметризованими запитами через `text()` та `UPDATE` оператор. Не забудьте на початку перевірити чи існує клієнт з таким номером в базі - це хороша практика.

Отримати всі колонки, які існують в таблиці ви можете наступним запитом
```
  SELECT COLUMN_NAME, DATA_TYPE
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_NAME = 'customers'
```

Запустіть функцію і продемонструйте її роботу, запустивши SELECT, який допоможе це зробити.



In [1]:
pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import datetime
import requests
import json
import os
import matplotlib.pyplot as plt

from dotenv import load_dotenv
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text, MetaData, Table
from sqlalchemy.orm import sessionmaker

In [3]:
# Примусово перезавантажуємо .env
load_dotenv(override=True)

print("Завантажений DB_HOST:", os.getenv("DB_HOST"))
print("Завантажений DB_PORT:", os.getenv("DB_PORT"))
print("Завантажений DB_USER:", os.getenv("DB_USER"))

Завантажений DB_HOST: 127.0.0.1
Завантажений DB_PORT: 3306
Завантажений DB_USER: root


In [5]:
# 1. Завантажуємо змінні середовища з .env
load_dotenv()

# 2. Формуємо URL і створюємо engine
db_url = f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(db_url)

In [6]:
employee_query = """
    SELECT COLUMN_NAME, DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME = 'customers';
"""
df_columns = pd.read_sql(text(employee_query), con=engine)

display(df_columns)

,COLUMN_NAME,DATA_TYPE
0,addressLine1,varchar
1,addressLine2,varchar
2,city,varchar
3,contactFirstName,varchar
4,contactLastName,varchar
5,country,varchar
6,creditLimit,decimal
7,customerName,varchar
8,customerNumber,int
9,phone,varchar


In [7]:

def update_customer_info(engine, customerNumber: int, new_phone: str = None, new_email: str = None):
    # 1. Перевіряємо, які колонки є в таблиці 'customers'
    cols_query = text("""
        SELECT COLUMN_NAME 
        FROM INFORMATION_SCHEMA.COLUMNS 
        WHERE TABLE_NAME = 'customers'
    """)
    
    # 2. Відкриваємо транзакцію для безпечного оновлення
    try:
        with engine.begin() as conn:
            # Отримуємо список усіх колонок
            existing_columns = [row[0] for row in conn.execute(cols_query).fetchall()]
            
            # КРОК A: Перевіряємо, чи існує клієнт
            check_query = text("SELECT customerName FROM customers WHERE customerNumber = :customerNumber")
            result = conn.execute(check_query, {'customerNumber': customerNumber})
            customer = result.fetchone()

            if not customer:
                print(f"❌ Клієнт з номером {customerNumber} не знайдений!")
                return False

            customer_name = customer[0]
            print(f"👤 Знайдено клієнта: {customer_name} (ID: {customerNumber})")

            # КРОК B: Формуємо оновлення телефону
            if new_phone:
                update_phone_query = text("""
                    UPDATE customers 
                    SET phone = :phone 
                    WHERE customerNumber = :customerNumber
                """)
                conn.execute(update_phone_query, {'phone': new_phone, 'customerNumber': customerNumber})
                print(f"  ✅ Телефон оновлено на: {new_phone}")

            # КРОК C: Оновлюємо email ТІЛЬКИ ЯКЩО таке поле існує в таблиці
            if new_email:
                if 'email' in existing_columns:
                    update_email_query = text("""
                        UPDATE customers 
                        SET email = :email 
                        WHERE customerNumber = :customerNumber
                    """)
                    conn.execute(update_email_query, {'email': new_email, 'customerNumber': customerNumber})
                    print(f"  ✅ Email оновлено на: {new_email}")
                else:
                    print("  ⚠️ Поле 'email' відсутнє в таблиці 'customers' (оновлення email пропущено).")

            print(f"🎉 Оновлення клієнта {customer_name} успішно завершено!")
            return True

    except Exception as e:
        print(f"❌ Помилка при оновленні: {e}")
        return False

In [8]:
# 1. Фіксуємо номер клієнта (наприклад, 103) та новий телефон
test_id = 103
test_phone = "+380991112233"
test_email = "test@example.com"

print("--- 1. ДАНІ ДО ОНОВЛЕННЯ ---")
df_before = pd.read_sql(
    text("SELECT customerNumber, customerName, phone FROM customers WHERE customerNumber = :id"),
    con=engine,
    params={'id': test_id}
)
display(df_before)

print("\n--- 2. ВИКОНАННЯ ФУНКЦІЇ ---")
update_customer_info(engine, customerNumber=test_id, new_phone=test_phone, new_email=test_email)

print("\n--- 3. ДАНІ ПІСЛЯ ОНОВЛЕННЯ ---")
df_after = pd.read_sql(
    text("SELECT customerNumber, customerName, phone FROM customers WHERE customerNumber = :id"),
    con=engine,
    params={'id': test_id}
)
display(df_after)

--- 1. ДАНІ ДО ОНОВЛЕННЯ ---


,customerNumber,customerName,phone
0,103,Atelier graphique,+380991112233



--- 2. ВИКОНАННЯ ФУНКЦІЇ ---
👤 Знайдено клієнта: Atelier graphique (ID: 103)
  ✅ Телефон оновлено на: +380991112233
  ⚠️ Поле 'email' відсутнє в таблиці 'customers' (оновлення email пропущено).
🎉 Оновлення клієнта Atelier graphique успішно завершено!

--- 3. ДАНІ ПІСЛЯ ОНОВЛЕННЯ ---


,customerNumber,customerName,phone
0,103,Atelier graphique,+380991112233


### Завдання 2: Створення нового замовлення з транзакцією (5 балів)

**Реалізуйте процес створення нового замовлення** з наступними кроками в одній транзакції:
- Створення запису в таблиці `orders`
- Додавання товарних позицій в `orderdetails`
- Перевірка наявності товарів на складі
- Зменшення кількості товарів на складі

Запустіть процес з тестовими даними і продемонструйте через SELECT, що процес успішно відпрацював і були виконані необхідні операції.




In [12]:

def create_new_order_transaction(
    engine, customer_number: int, product_code: str, quantity: int, price_each: float
):
    """Створює нове замовлення, додає деталі та списує товар зі складу в одній транзакції."""
    today = date.today().strftime("%Y-%m-%d")

    try:
        with engine.begin() as conn:

           
            # Перевіряємо наявність товару на складі
            
            check_stock_query = text("""
                SELECT productName, quantityInStock 
                FROM products 
                WHERE productCode = :product_code
            """)
            product = conn.execute(
                check_stock_query, {"product_code": product_code}
            ).fetchone()

            if not product:
                print(f"❌ Товар {product_code} не знайдено в базі!")
                return False

            product_name, current_stock = product[0], product[1]

            if current_stock < quantity:
                print(
                    f"❌ Недостатньо товару '{product_name}' на складі! "
                    f"Запитувано: {quantity}, Є в наявності: {current_stock}"
                )
                print("🔄 Транзакцію скасовано (ROLLBACK).")
                return False

            print(
                f"📦 Товар '{product_name}' є в наявності ({current_stock} шт.). Починаємо оформлення..."
            )

            
            #Генеруємо новий orderNumber та створюємо замовлення в `orders`
     
            # Знаходимо максимальний застарілий orderNumber і додаємо 1
            max_id_query = text(
                "SELECT COALESCE(MAX(orderNumber), 10000) + 1 FROM orders"
            )
            new_order_number = conn.execute(max_id_query).scalar()

            insert_order_query = text("""
                INSERT INTO orders (orderNumber, orderDate, requiredDate, status, customerNumber)
                VALUES (:order_num, :order_date, :req_date, 'In Process', :customer_num)
            """)
            conn.execute(
                insert_order_query,
                {
                    "order_num": new_order_number,
                    "order_date": today,
                    "req_date": today,
                    "customer_num": customer_number,
                },
            )
            print(f"  ✅ Крок 1: Створено замовлення №{new_order_number}")

            # Додаємо товар у `orderdetails`

            insert_details_query = text("""
                INSERT INTO orderdetails (orderNumber, productCode, quantityOrdered, priceEach, orderLineNumber)
                VALUES (:order_num, :product_code, :qty, :price, 1)
            """)
            conn.execute(
                insert_details_query,
                {
                    "order_num": new_order_number,
                    "product_code": product_code,
                    "qty": quantity,
                    "price": price_each,
                },
            )
            print(
                f"  ✅ Крок 2: Додано позицію у замовлення ({quantity} шт. х ${price_each})"
            )

            
            #  Зменшуємо кількість товару на складі в `products`
            
            update_stock_query = text("""
                UPDATE products
                SET quantityInStock = quantityInStock - :qty
                WHERE productCode = :product_code
            """)
            conn.execute(
                update_stock_query,
                {"qty": quantity, "product_code": product_code},
            )
            print(
                f"  ✅ Крок 3: Списано {quantity} шт. зі складу. Залишок: {current_stock - quantity} шт."
            )

            print(
                f"\n🎉 Замовлення №{new_order_number} успішно створено! (COMMIT)"
            )
            return new_order_number

    except Exception as e:
        print(f"❌ Помилка під час створення замовлення: {e}")
        print("🔄 Всі зміни автоматично скасовано (ROLLBACK)")
        return False

In [14]:
# Тестові дані для замовлення
test_customer_id = 103
test_product_code = "S10_1678"  
buy_quantity = 5
item_price = 95.00

print(" 1.КІЛЬКІСТЬ ТОВАРУ НА СКЛАДІ ДО КУПІВЛІ")
df_stock_before = pd.read_sql(
    text(
        "SELECT productCode, productName, quantityInStock FROM products WHERE productCode = :code"
    ),
    con=engine,
    params={"code": test_product_code},
)
display(df_stock_before)

print("\n 2. ЗАПУСК ТРАНЗАКЦІЇ СТВОРЕННЯ ЗАМОВЛЕННЯ ===")
created_order_id = create_new_order_transaction(
    engine,
    customer_number=test_customer_id,
    product_code=test_product_code,
    quantity=buy_quantity,
    price_each=item_price,
)

if created_order_id:
    print("\n 3. ПЕРЕВІРКА СТВОРЕНОГО ЗАМОВЛЕННЯ (SELECT ORDERS) ")
    df_order = pd.read_sql(
        text(
            "SELECT orderNumber, orderDate, status, customerNumber FROM orders WHERE orderNumber = :id"
        ),
        con=engine,
        params={"id": created_order_id},
    )
    display(df_order)

    print(
        "\n 4. ПЕРЕВІРКА ПОЗИЦІЙ В ЗАМОВЛЕННІ (SELECT ORDERDETAILS) "
    )
    df_details = pd.read_sql(
        text(
            "SELECT orderNumber, productCode, quantityOrdered, priceEach FROM orderdetails WHERE orderNumber = :id"
        ),
        con=engine,
        params={"id": created_order_id},
    )
    display(df_details)

    print("\n=== 5. КІЛЬКІСТЬ ТОВАРУ НА СКЛАДІ ПІСЛЯ КУПІВЛІ ===")
    df_stock_after = pd.read_sql(
        text(
            "SELECT productCode, productName, quantityInStock FROM products WHERE productCode = :code"
        ),
        con=engine,
        params={"code": test_product_code},
    )
    display(df_stock_after)

 1.КІЛЬКІСТЬ ТОВАРУ НА СКЛАДІ ДО КУПІВЛІ


,productCode,productName,quantityInStock
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7923



 2. ЗАПУСК ТРАНЗАКЦІЇ СТВОРЕННЯ ЗАМОВЛЕННЯ ===
📦 Товар '1969 Harley Davidson Ultimate Chopper' є в наявності (7923 шт.). Починаємо оформлення...
  ✅ Крок 1: Створено замовлення №10428
  ✅ Крок 2: Додано позицію у замовлення (5 шт. х $95.0)
  ✅ Крок 3: Списано 5 шт. зі складу. Залишок: 7918 шт.

🎉 Замовлення №10428 успішно створено! (COMMIT)

 3. ПЕРЕВІРКА СТВОРЕНОГО ЗАМОВЛЕННЯ (SELECT ORDERS) 


,orderNumber,orderDate,status,customerNumber
0,10428,2026-10-02,In Process,103



 4. ПЕРЕВІРКА ПОЗИЦІЙ В ЗАМОВЛЕННІ (SELECT ORDERDETAILS) 


,orderNumber,productCode,quantityOrdered,priceEach
0,10428,S10_1678,5,95.0



=== 5. КІЛЬКІСТЬ ТОВАРУ НА СКЛАДІ ПІСЛЯ КУПІВЛІ ===


,productCode,productName,quantityInStock
0,S10_1678,1969 Harley Davidson Ultimate Chopper,7918
